In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient
from langchain_community.utilities import SQLDatabase

tavily_client = TavilyClient()

db = SQLDatabase.from_uri("sqlite:///Chinook.db")


@tool
def web_search(query: str) -> Dict[str, Any]:

    """Search the web for information"""

    return tavily_client.search(query)

@tool
def sql_query(query: str) -> str:

    """Obtain information from the database using SQL queries"""

    try:
        return db.run(query)
    except Exception as e:
        return f"Error: {e}"

In [4]:
from dataclasses import dataclass

@dataclass
class UserRole:
    user_role: str = "external"

In [5]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Callable

@wrap_model_call
def dynamic_tool_call(request: ModelRequest, 
handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:

    """Dynamically call tools based on the runtime context"""

    user_role = request.runtime.context.user_role
    
    if user_role == "internal":
        pass # internal users get access to all tools
    else:
        tools = [web_search] # external users only get access to web search
        request = request.override(tools=tools) 

    return handler(request)

In [8]:
from langchain.agents import create_agent

agent = create_agent(
    model="ollama:qwen2.5:3b",
    tools=[web_search, sql_query],
    middleware=[dynamic_tool_call], # type: ignore
    context_schema=UserRole
) # type: ignore

In [9]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="How many artists are in the database?")]},
    context={"user_role": "external"}
)

print(response["messages"][-1].content)

Based on the information provided from the search results, there are approximately 2,931,346 artists in the database. This number comes from a link to MusicBrainz's statistics page, which is one of the most accurate and official types of databases for these entities due to its direct communication with the artists, record labels, distributors, legal teams, publishers, and governing bodies.

While other sources provide information on different counts (like 12 million tracks or 9 million tracks), none specifically mention artist counts. Therefore, we will rely on the most authoritative source provided by MusicBrainz. 

Let me know if you need any further assistance!
